# Module 6 Notebook: Clustering and Customer Segmentation
### Machine Learning for Business Insights: course template

**The business question:** *What natural groups of customers do we have, and how should we treat each group differently?*

There is no "right answer" column here. This is **unsupervised** learning: K-means groups customers who behave alike, and **you** give each group a name and an action. That last step is where segmentation earns its keep.

📘 **Companion lesson:** [Module 6: Clustering and Customer Segmentation](../lessons/module_06_clustering.md) (Lessons 6.3 to 6.9 map directly onto the sections below).

| Section | What happens | Lesson |
|---|---|---|
| 1. Light EDA | Blanks, skew, and which columns need a log | 6.3 |
| 2. Prep pipeline | Fill gaps → log1p for skewed columns → scale | 6.3 |
| 3. How many segments? | Elbow + silhouette + minimum segment size | 6.5 |
| 4. Fit K-means | Final segmentation (MiniBatchKMeans on large data) | 6.4 |
| 5. Stability checks | Restarts, resamples, and K ± 1 | 6.9 |
| 6. Segment profiles | Original units, index vs. average, plain-language descriptions | 6.8 |
| 7. Action table | Suggested name + suggested action per segment | 6.8 |
| 8. PCA map | 2-D picture of the segments | 6.7 |
| 9. Rulebook | Shallow decision tree that staff can apply without code | 6.8 |
| 10–12 | Small vs. large data, save, checklist and next steps | |

**How to use it**
- **Edit only the CONFIG cell, then run top to bottom** (Runtime → Run all in Colab)
- Leave `CSV_PATH = None` the first time to run on a built-in demo of 900 retail customers
- Read the printed notes after each step, since they tell you what to look for

> Demo data is **synthetic**. The workflow is real; the numbers it prints are for practice only.

In [ ]:
# ============================================================
# IMPORTS (all preinstalled in Google Colab; needs scikit-learn 1.2+)
# Local Jupyter: pip install pandas numpy scikit-learn matplotlib seaborn joblib
# ============================================================
import warnings
warnings.filterwarnings("ignore")        # keeps output readable for beginners

import numpy as np                       # math on arrays of numbers
import pandas as pd                      # tables of data (like Excel in Python)
import matplotlib.pyplot as plt          # charts
import seaborn as sns                    # nicer charts
from IPython.display import display

import sklearn
from sklearn.pipeline import Pipeline                      # chains prep steps + model
from sklearn.compose import ColumnTransformer              # different prep per column group
from sklearn.impute import SimpleImputer                   # fills blanks
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.cluster import KMeans, MiniBatchKMeans        # the clustering algorithms
from sklearn.decomposition import PCA                      # squeezes many columns into 2 for a chart
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

major, minor = (int(p) for p in sklearn.__version__.split(".")[:2])
assert (major, minor) >= (1, 2), f"Please upgrade scikit-learn to 1.2+ (you have {sklearn.__version__})"
print(f"Libraries loaded. scikit-learn {sklearn.__version__}")

In [ ]:
# ============================================================
# CONFIG: THE ONLY CELL YOU NEED TO EDIT
# ============================================================

# --- 1. Where is your data? ---------------------------------
# e.g. "/content/customers.csv" in Colab (upload via the folder icon first).
# Leave as None to use the built-in synthetic demo customers.
CSV_PATH = None

# --- 2. Which columns? ---------------------------------------
ID_COLUMN = "customer_id"            # unique customer ID (never used to form segments); None if you have none

# Behaviour columns used to FORM the segments. Numbers only, 3–8 columns,
# each with a business reason. No IDs, no outcome columns (see Lesson 6.8).
CLUSTER_COLUMNS = ["annual_spend", "orders_12m", "avg_order_value", "days_since_last_order",
                   "pct_orders_discounted", "support_contacts_12m", "tenure_months"]

# Skewed money/count columns (long right tail). A log transform stops a few huge
# customers from dominating. Must be a subset of CLUSTER_COLUMNS, no negative values.
LOG_COLUMNS = ["annual_spend", "orders_12m", "avg_order_value", "support_contacts_12m"]

# Optional columns used ONLY to DESCRIBE segments afterwards (never to form them).
# Outcomes (margin, NPS, churned) and labels (channel, region) belong here. Use [] if none.
PROFILE_COLUMNS = ["gross_margin_12m", "nps_score", "preferred_channel", "region"]

VALUE_COLUMN = "annual_spend"        # used for each segment's share of revenue (None to skip)

# --- 3. How many segments to consider? ----------------------
K_RANGE = range(2, 9)                # segment counts to try
MIN_K_FOR_BUSINESS = 3               # fewer than 3 segments is rarely actionable
MIN_SEGMENT_PCT = 5                  # every segment must hold at least this % of customers
K_OVERRIDE = None                    # set e.g. 4 to replace the automatic choice

# --- 4. Data size and reproducibility -----------------------
DATA_SIZE_MODE = "auto"              # "auto", "small", or "large"
                                     # large -> MiniBatchKMeans, sampled silhouette, fewer resamples
SILHOUETTE_SAMPLE = 3_000            # rows sampled for the silhouette score in large mode
RANDOM_STATE = 42                    # fixed seed so results repeat exactly

# --- 5. Outputs ---------------------------------------------
SAVE_OUTPUTS = True                  # write customer_segments.csv + segmentation_model.joblib

In [ ]:
# ============================================================
# LOAD DATA (or build a synthetic customer base)
# The demo is SYNTHETIC: do not draw real conclusions from it.
# ============================================================
def make_demo_customers(n=900, seed=RANDOM_STATE):
    """Synthetic retail customers drawn from four hidden personas:
    VIP loyalists, bargain hunters, new customers, and lapsing customers.
    K-means never sees these labels; we check later whether it rediscovers them."""
    rng = np.random.default_rng(seed)
    persona = rng.choice(["vip", "bargain", "new", "lapsing"], n, p=[0.15, 0.30, 0.30, 0.25])
    # p(vip=..., bargain=...) returns, for each customer, the value for their persona
    p = lambda **kw: np.select([persona == k for k in kw], list(kw.values()))

    orders = np.maximum(1, rng.poisson(p(vip=24, bargain=10, new=3, lapsing=5)))
    aov = np.round(rng.lognormal(np.log(p(vip=140, bargain=45, new=70, lapsing=80)), 0.3), 2)
    spend = np.round(orders * aov, 2)
    disc = np.clip(rng.normal(p(vip=0.10, bargain=0.70, new=0.30, lapsing=0.25), 0.10), 0, 1).round(2)
    df = pd.DataFrame({
        "customer_id": np.arange(20_001, 20_001 + n),
        "orders_12m": orders,
        "avg_order_value": aov,
        "annual_spend": spend,
        "days_since_last_order": np.clip(rng.normal(p(vip=12, bargain=30, new=25, lapsing=170),
                                                    p(vip=8, bargain=15, new=15, lapsing=50)), 0, 365).round(),
        "pct_orders_discounted": disc,
        "support_contacts_12m": rng.poisson(p(vip=2, bargain=1, new=1.5, lapsing=3)).astype(float),
        "tenure_months": np.clip(rng.normal(p(vip=48, bargain=30, new=4, lapsing=36),
                                            p(vip=15, bargain=12, new=2, lapsing=14)), 1, 120).round(),
        # Profile-only columns (outcomes and labels, NOT used to form segments)
        "gross_margin_12m": np.round(spend * np.clip(0.42 - 0.30 * disc + rng.normal(0, 0.03, n), 0.02, 0.6), 2),
        "nps_score": np.clip(np.round(rng.normal(p(vip=9, bargain=7, new=7.5, lapsing=4.5), 1.5)), 0, 10),
        "preferred_channel": np.where(rng.random(n) < p(vip=0.3, bargain=0.8, new=0.6, lapsing=0.5), "online", "store"),
        "region": rng.choice(["north", "south", "east", "west"], n),
    })
    df.loc[rng.random(n) < 0.03, "support_contacts_12m"] = np.nan   # a few blanks, like real data
    df.loc[rng.random(n) < 0.02, "tenure_months"] = np.nan
    df["_hidden_persona"] = persona          # demo check only; never used for clustering
    return df

if CSV_PATH is None:
    df = make_demo_customers()
    print("Using SYNTHETIC demo data (CSV_PATH is None).")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {CSV_PATH}")
print(f"{len(df):,} customers x {df.shape[1]} columns")
df.head()

In [ ]:
# ============================================================
# CONFIG CHECK
# Catches typos and mistakes before anything else breaks.
# ============================================================
need = CLUSTER_COLUMNS + PROFILE_COLUMNS + ([VALUE_COLUMN] if VALUE_COLUMN else [])
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"These CONFIG columns are not in your data: {missing}\nYour columns: {list(df.columns)}")
if not set(LOG_COLUMNS) <= set(CLUSTER_COLUMNS):
    raise ValueError(f"LOG_COLUMNS must also be in CLUSTER_COLUMNS: {set(LOG_COLUMNS) - set(CLUSTER_COLUMNS)}")
overlap = set(CLUSTER_COLUMNS) & set(PROFILE_COLUMNS + ([ID_COLUMN] if ID_COLUMN else []))
if overlap:
    raise ValueError(f"These are listed as clustering inputs AND as ID/profile columns: {overlap}")
if VALUE_COLUMN and VALUE_COLUMN not in CLUSTER_COLUMNS and VALUE_COLUMN not in PROFILE_COLUMNS:
    print(f"Note: VALUE_COLUMN '{VALUE_COLUMN}' is only used for revenue share.")

# Clustering columns must be numbers (e.g. "$1,200" stored as text gets cleaned here)
X = df[CLUSTER_COLUMNS].copy()
for c in CLUSTER_COLUMNS:
    if X[c].dtype == object:
        X[c] = pd.to_numeric(X[c].astype(str).str.replace(r"[$,%\s]", "", regex=True), errors="coerce")
X = X.astype(float)

neg = [c for c in LOG_COLUMNS if (X[c] < 0).any()]
if neg:
    raise ValueError(f"Log columns can't contain negative values: {neg}. Remove them from LOG_COLUMNS.")
if ID_COLUMN and ID_COLUMN in df.columns and df[ID_COLUMN].duplicated().any():
    print(f"⚠ {df[ID_COLUMN].duplicated().sum()} duplicate customer IDs: one row per customer is expected.")
print("Config check passed ✓")

## Section 1: Light EDA: Blanks, Skew, and Size

Clustering is only as good as its inputs. Three quick checks:
1. **Blanks:** a few are fine (we fill them with the median); a column that is mostly blank should go
2. **Skew:** money and count columns usually have a long right tail. A few huge customers would otherwise pull a segment all of their own. **Skew above ~1 → add the column to `LOG_COLUMNS`**
3. **Size:** decides how cautious the rest of the notebook is

In [ ]:
# ============================================================
# DATA QUALITY + SKEW REPORT
# ============================================================
eda = pd.DataFrame({
    "missing_%": X.isna().mean() * 100,
    "mean": X.mean(), "median": X.median(), "min": X.min(), "max": X.max(),
    "skew": X.skew(),
    "in LOG_COLUMNS": [c in LOG_COLUMNS for c in CLUSTER_COLUMNS],
})
display(eda.round(2))

suggest = [c for c in CLUSTER_COLUMNS if eda.loc[c, "skew"] > 1 and c not in LOG_COLUMNS and X[c].min() >= 0]
if suggest:
    print(f"💡 Consider adding to LOG_COLUMNS (skew > 1): {suggest}")
mostly_blank = eda.index[eda["missing_%"] > 40].tolist()
if mostly_blank:
    print(f"⚠ Mostly blank, consider dropping: {mostly_blank}")
print(f"Duplicate rows: {df.duplicated().sum()}")

# Histograms: long right tails = log candidates
n_cols = 4
n_rows_plot = int(np.ceil(len(CLUSTER_COLUMNS) / n_cols))
X.hist(bins=30, figsize=(14, 3 * n_rows_plot), layout=(n_rows_plot, n_cols), color="slateblue")
plt.suptitle("Clustering columns (long right tails → log)"); plt.tight_layout(); plt.show()

# Correlation: two columns above ~0.8 measure the same thing twice and double its weight
plt.figure(figsize=(7, 5.5))
sns.heatmap(X.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Correlation between clustering columns"); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# DATA SIZE CHECK
# Small data -> standard K-means with many restarts and more stability checks.
# Large data -> MiniBatchKMeans (fast), sampled silhouette, fewer resamples.
# ============================================================
n_rows = len(X)
if DATA_SIZE_MODE == "auto":
    MODE = "small" if n_rows < 20_000 else "large"
else:
    MODE = DATA_SIZE_MODE
assert MODE in ("small", "large"), "DATA_SIZE_MODE must be 'auto', 'small', or 'large'"

N_RESAMPLES = 10 if MODE == "small" else 5       # stability re-fits
SIL_SAMPLE = SILHOUETTE_SAMPLE if (MODE == "large" and n_rows > SILHOUETTE_SAMPLE) else None

def make_kmeans(k, seed=RANDOM_STATE, init="k-means++"):
    """One place that decides which K-means flavour to use."""
    n_init = 1 if not isinstance(init, str) else None
    if MODE == "large":
        return MiniBatchKMeans(n_clusters=k, init=init, n_init=n_init or 5, batch_size=2_048,
                               random_state=seed)
    return KMeans(n_clusters=k, init=init, n_init=n_init or 20, random_state=seed)

print(f"{n_rows:,} customers, {len(CLUSTER_COLUMNS)} clustering columns -> MODE = '{MODE}' "
      f"({type(make_kmeans(2)).__name__})")
if n_rows < 300:
    print("⚠ Under ~300 customers: keep K at 3–4 and treat segments as a first draft.")

## Section 2: Leakage-Safe Prep Pipeline

K-means groups customers by **distance**, so every column must be on the same scale. Otherwise "annual spend in dollars" would drown out "share of orders discounted" (Lesson 6.3).

| Column group | Steps |
|---|---|
| `LOG_COLUMNS` (skewed) | Fill blanks with the median → `log1p` → standardize |
| Other clustering columns | Fill blanks with the median → standardize |

Bundling these steps with the model in a **Pipeline** means new customers are prepared *exactly* the same way later, with the fill values and scaling learned from this data.

In [ ]:
# ============================================================
# PREPROCESSOR
# ============================================================
other_cols = [c for c in CLUSTER_COLUMNS if c not in LOG_COLUMNS]

transformers = []
if LOG_COLUMNS:
    transformers.append(("log", Pipeline([
        ("impute", SimpleImputer(strategy="median")),                                # blanks -> middle value
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),    # tame the long tail
        ("scale", StandardScaler()),                                                  # mean 0, typical step 1
    ]), LOG_COLUMNS))
if other_cols:
    transformers.append(("plain", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), other_cols))
prep = ColumnTransformer(transformers, verbose_feature_names_out=False)

Z = prep.fit_transform(X)          # Z = prepared matrix K-means will see
print(f"Prepared matrix: {Z.shape[0]:,} rows x {Z.shape[1]} columns")
print("Every column now has mean ≈ 0 and spread ≈ 1:")
display(pd.DataFrame(Z, columns=prep.get_feature_names_out()).describe().loc[["mean", "std"]].round(2))

## Section 3: How Many Segments? Elbow + Silhouette + Size

Three lenses (Lesson 6.5):
- **Elbow (inertia):** how tightly customers sit around their segment centre. It always improves with more segments; look for where the improvement **slows down**.
- **Silhouette (−1 to 1):** how clearly each customer belongs to its own segment rather than the next one. Above 0.5 = clearly separated; 0.25–0.5 = reasonable (common in business data); below 0.25 = weak.
- **Smallest segment:** a segment with 1% of customers rarely justifies its own strategy. K values whose smallest segment is under `MIN_SEGMENT_PCT` are ruled out.

**Automatic rule:** the highest silhouette among K ≥ `MIN_K_FOR_BUSINESS` whose smallest segment is ≥ `MIN_SEGMENT_PCT`. The elbow is a second opinion. **Business judgment has the final word:** set `K_OVERRIDE` if your team can only run, say, three campaigns.

In [ ]:
# ============================================================
# K SCAN
# ============================================================
rows, scan_labels = [], {}
for k in K_RANGE:
    km = make_kmeans(k).fit(Z)
    scan_labels[k] = km.labels_
    shares = np.bincount(km.labels_, minlength=k) / len(Z) * 100
    rows.append({"k": k, "inertia": km.inertia_,
                 "silhouette": silhouette_score(Z, km.labels_, sample_size=SIL_SAMPLE, random_state=RANDOM_STATE),
                 "smallest_segment_%": shares.min()})
k_tbl = pd.DataFrame(rows)

# Elbow point ("kneedle" idea): the K farthest below the straight line from the first to the last point
xs = (k_tbl["k"] - k_tbl["k"].min()) / max(k_tbl["k"].max() - k_tbl["k"].min(), 1)
ys = (k_tbl["inertia"] - k_tbl["inertia"].min()) / max(k_tbl["inertia"].max() - k_tbl["inertia"].min(), 1e-12)
elbow_k = int(k_tbl.loc[((1 - xs) - ys).idxmax(), "k"])

k_tbl["eligible"] = (k_tbl["k"] >= MIN_K_FOR_BUSINESS) & (k_tbl["smallest_segment_%"] >= MIN_SEGMENT_PCT)
eligible = k_tbl[k_tbl["eligible"]]
if len(eligible):
    sil_k = int(eligible.loc[eligible["silhouette"].idxmax(), "k"])
    auto_k, auto_reason = sil_k, "best silhouette among eligible K"
else:
    sil_k = None
    auto_k, auto_reason = max(elbow_k, MIN_K_FOR_BUSINESS), "no K passed the size rule, so falling back to the elbow"
K = int(K_OVERRIDE) if K_OVERRIDE else auto_k

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
axes[0].plot(k_tbl["k"], k_tbl["inertia"], "o-"); axes[0].axvline(elbow_k, ls="--", c="grey")
axes[0].set_title(f"Elbow (suggests K={elbow_k})"); axes[0].set_xlabel("K"); axes[0].set_ylabel("inertia")
axes[1].plot(k_tbl["k"], k_tbl["silhouette"], "o-", c="darkorange"); axes[1].axvline(K, ls="--", c="grey")
axes[1].axhline(0.25, ls=":", c="red"); axes[1].set_title(f"Silhouette (chosen K={K})"); axes[1].set_xlabel("K")
axes[2].bar(k_tbl["k"], k_tbl["smallest_segment_%"], color=np.where(k_tbl["eligible"], "seagreen", "lightgrey"))
axes[2].axhline(MIN_SEGMENT_PCT, ls="--", c="red"); axes[2].set_title("Smallest segment % (grey = ruled out)")
axes[2].set_xlabel("K")
plt.tight_layout(); plt.show()
display(k_tbl.round(3))

print(f"Elbow suggests K={elbow_k}; silhouette (eligible) suggests K={sil_k} ({auto_reason}).")
print(f"Using K = {K}" + (" (set by K_OVERRIDE in CONFIG)" if K_OVERRIDE else ""))
best_sil = k_tbl.loc[k_tbl["k"] == K, "silhouette"].iloc[0] if K in K_RANGE else np.nan
if best_sil == best_sil:   # not NaN
    print("Reading: " + ("clear structure." if best_sil > 0.5 else
                         "reasonable structure with some overlap (normal for customers)." if best_sil > 0.25 else
                         "weak structure; segments blur together. Consider fewer columns or simple RFM rules."))

## Section 4: Fit the Final Segmentation

We fit the full pipeline (prep + K-means) so it can later score new customers with `.predict()`.
To make tables easy to read, segments are **renumbered by value**: segment 0 has the highest average `VALUE_COLUMN`.

In [ ]:
# ============================================================
# FINAL MODEL
# ============================================================
first = make_kmeans(K).fit(Z)

# Renumber segments by value (highest first). We refit starting exactly from the
# reordered centres, which gives the same groups with tidy numbering.
order_col = VALUE_COLUMN if VALUE_COLUMN in CLUSTER_COLUMNS else CLUSTER_COLUMNS[0]
seg_value = X[order_col].groupby(first.labels_).mean()
order = seg_value.sort_values(ascending=False).index.to_numpy()
seg_model = Pipeline([("prep", prep),
                      ("kmeans", make_kmeans(K, init=first.cluster_centers_[order]))]).fit(X)
labels = seg_model.predict(X)
SEGMENTS = list(range(K))
size = pd.Series(labels).value_counts().reindex(SEGMENTS, fill_value=0)

print(f"Fitted {type(seg_model.named_steps['kmeans']).__name__} with K = {K}")
print(f"Agreement with the first fit (should be ~1.00): ARI {adjusted_rand_score(first.labels_, labels):.2f}")
display(pd.DataFrame({"customers": size, "share_%": (size / size.sum() * 100).round(1)}).rename_axis("segment"))

## Section 5: Are the Segments Real? Stability Checks

K-means always produces segments, even on random noise (Lesson 6.9). We check whether the **same** groups come back when things change. Agreement is measured with the **Adjusted Rand Index (ARI)**: 1.0 = identical groupings, 0 = no better than chance.

| Check | What changes | Reading |
|---|---|---|
| Restarts | Random starting points | Should be ≥ 0.90; lower means K-means is finding different answers |
| Resamples | Fit on a bootstrap sample of customers, then assign **everyone** | ≥ 0.80 stable · 0.60–0.80 broadly OK, fine details shaky · < 0.60 don't build strategy on it |
| Per segment | How often each segment reappears in the resamples (Jaccard overlap) | A segment below ~0.6 is borderline: consider merging it |
| K ± 1 | One segment fewer / more | Shows which segments merge or split; useful when deciding between two K values |

In [ ]:
# ============================================================
# STABILITY: RESTARTS + BOOTSTRAP RESAMPLES + PER-SEGMENT JACCARD
# ============================================================
rng = np.random.default_rng(RANDOM_STATE)

restart_ari = [adjusted_rand_score(labels, make_kmeans(K, seed=s).fit(Z).labels_)
               for s in range(1, N_RESAMPLES + 1)]

def best_jaccard(ref, other, k):
    """For each reference segment: overlap with its best-matching segment in another run."""
    out = []
    for s in range(k):
        a = ref == s
        out.append(max(((a & (other == t)).sum() / max((a | (other == t)).sum(), 1)) for t in np.unique(other)))
    return out

boot_ari, jac = [], []
for b in range(N_RESAMPLES):
    idx = rng.integers(0, len(Z), len(Z))                       # sample customers WITH replacement
    km_b = make_kmeans(K, seed=RANDOM_STATE + b).fit(Z[idx])
    lab_b = km_b.predict(Z)                                     # assign EVERY customer with the resampled model
    boot_ari.append(adjusted_rand_score(labels, lab_b))
    jac.append(best_jaccard(labels, lab_b, K))
seg_stability = pd.Series(np.mean(jac, axis=0), index=SEGMENTS).round(2)

def verdict(v):
    return "stable ✓" if v >= 0.8 else ("broadly OK" if v >= 0.6 else "unstable ⚠")

stab = pd.DataFrame({
    "check": ["Restarts (different seeds)", "Bootstrap resamples"],
    "mean ARI": [np.mean(restart_ari), np.mean(boot_ari)],
    "worst ARI": [np.min(restart_ari), np.min(boot_ari)],
})
stab["verdict"] = stab["worst ARI"].apply(verdict)
display(stab.round(2))
print("Per-segment stability (Jaccard, 1.0 = always reappears):")
display(seg_stability.to_frame("jaccard").assign(verdict=seg_stability.apply(verdict)).rename_axis("segment"))

STABILITY_SCORE = float(np.mean(boot_ari))
if np.min(boot_ari) >= 0.8:
    print("Reading: stable. Safe to build actions on these segments.")
elif np.min(boot_ari) >= 0.6:
    print("Reading: broadly stable. Act on the big picture, be careful with fine details.")
else:
    print("Reading: unstable. Try a smaller K, fewer/better columns, or more data before acting.")

if "_hidden_persona" in df.columns:
    print(f"\n(Demo only) Match with the hidden personas, ARI: {adjusted_rand_score(df['_hidden_persona'], labels):.2f}")
    display(pd.crosstab(df["_hidden_persona"], labels, rownames=["hidden persona"], colnames=["segment"]))

In [ ]:
# ============================================================
# K ± 1: WHICH SEGMENTS MERGE OR SPLIT?
# Each table shows where today's segments (rows) go with one segment fewer or more (columns).
# A row spread over two columns = that segment splits; two rows landing in one column = they merge.
# ============================================================
for k_alt in (K - 1, K + 1):
    if k_alt < 2:
        continue
    alt = scan_labels[k_alt] if k_alt in scan_labels else make_kmeans(k_alt).fit(Z).labels_
    print(f"K={K} vs K={k_alt}: ARI {adjusted_rand_score(labels, alt):.2f}")
    display(pd.crosstab(labels, alt, rownames=[f"segment (K={K})"], colnames=[f"K={k_alt}"]))

## Section 6: Segment Profiles in Plain Business Terms

Profiles are shown in **original units** (dollars, days, orders), never in scaled units (Lesson 6.8 pitfall).
The **index** compares each segment's average with the overall average: **100 = average customer**, 150 = 50% above average, 50 = half the average. Indexes make very different columns easy to read side by side.

`PROFILE_COLUMNS` (margin, satisfaction, channel, region) were **not** used to form the segments. If segments still differ on them, that's good evidence the segments are meaningful.

In [ ]:
# ============================================================
# PROFILE TABLES (original units)
# ============================================================
prof_mean = X.groupby(labels).mean().reindex(SEGMENTS)
prof_median = X.groupby(labels).median().reindex(SEGMENTS)
overall = X.mean()
index = (prof_mean / overall * 100).round(0)

summary = prof_mean.copy()
summary.insert(0, "customers", size)
summary.insert(1, "share_%", (size / size.sum() * 100).round(1))
if VALUE_COLUMN:
    value = pd.to_numeric(df[VALUE_COLUMN], errors="coerce")
    rev = value.groupby(labels).sum().reindex(SEGMENTS)
    summary.insert(2, "revenue_share_%", (rev / rev.sum() * 100).round(1))
summary.loc["ALL"] = [len(X), 100.0] + ([100.0] if VALUE_COLUMN else []) + overall.tolist()

print("Segment AVERAGES (original units); last row = all customers")
display(summary.rename_axis("segment"))
print("Segment MEDIANS (less affected by a few extreme customers)")
display(prof_median.rename_axis("segment"))

plt.figure(figsize=(11, 0.6 * K + 2.2))
sns.heatmap(index, annot=True, fmt=".0f", cmap="RdBu_r", center=100, cbar_kws={"label": "index vs average"})
plt.title("Segment profile index (100 = average customer)"); plt.ylabel("segment"); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# PROFILE-ONLY COLUMNS (not used to form segments)
# ============================================================
num_prof = [c for c in PROFILE_COLUMNS if pd.api.types.is_numeric_dtype(df[c])]
cat_prof = [c for c in PROFILE_COLUMNS if c not in num_prof]
if num_prof:
    print("Outcome columns by segment (averages):")
    display(df[num_prof].groupby(labels).mean().reindex(SEGMENTS).rename_axis("segment"))
for c in cat_prof:
    display(pd.crosstab(labels, df[c], normalize="index").round(2).rename_axis(index="segment", columns=c))
if not PROFILE_COLUMNS:
    print("No PROFILE_COLUMNS set. Outcomes like margin or satisfaction make profiles much richer.")

In [ ]:
# ============================================================
# AUTO-DRAFTED DESCRIPTIONS
# The 2 most "above average" and 2 most "below average" traits per segment.
# Treat these as a first draft: rewrite them in your team's language.
# ============================================================
def describe(seg):
    row = index.loc[seg]
    hi = [f"high {c.replace('_', ' ')} ({row[c]:.0f})" for c in row.sort_values(ascending=False).index[:2] if row[c] > 115]
    lo = [f"low {c.replace('_', ' ')} ({row[c]:.0f})" for c in row.sort_values().index[:2] if row[c] < 85]
    return ", ".join(hi + lo) or "close to average on everything"

descriptions = {s: describe(s) for s in SEGMENTS}
for s in SEGMENTS:
    extra = f", {summary.loc[s, 'revenue_share_%']:.0f}% of revenue" if VALUE_COLUMN else ""
    print(f"Segment {s} ({size[s]:,} customers, {size[s] / size.sum():.0%}{extra}): {descriptions[s]}")

## Section 7: Action Table: Name Each Segment and Decide What to Do

A segment without an action is just a chart. The rules below turn the profile **index** into a suggested **name** and **action**.
They use the demo column names, so **edit them to match your columns and strategy**. Rules are checked top to bottom; the first match wins. Segments that match nothing are called "Core / steady".

In [ ]:
# ============================================================
# ACTION RULES (edit me)
# (column, condition on the index, suggested name, suggested action)
# ============================================================
ACTION_RULES = [
    ("days_since_last_order", lambda v: v > 150, "Lapsing",          "Win-back offer within 2 weeks; ask why they left"),
    ("annual_spend",          lambda v: v > 150, "VIP / high value", "Loyalty perks, early access, account manager; avoid blanket discounts"),
    ("pct_orders_discounted", lambda v: v > 140, "Bargain hunters",  "Targeted promos only; test smaller discounts to protect margin"),
    ("tenure_months",         lambda v: v < 50,  "New customers",    "Onboarding series and a second-purchase incentive"),
]
DEFAULT_NAME, DEFAULT_ACTION = "Core / steady", "Maintain; cross-sell related products"

actions, used = [], {}
for s in SEGMENTS:
    name, action = DEFAULT_NAME, DEFAULT_ACTION
    for col, cond, n, a in ACTION_RULES:
        if col in index.columns and cond(index.loc[s, col]):
            name, action = n, a
            break
    used[name] = used.get(name, 0) + 1
    if used[name] > 1:                      # keep names unique, e.g. "Core / steady (2)"
        name = f"{name} ({used[name]})"
    actions.append({"segment": s, "suggested_name": name, "customers": size[s],
                    "share_%": round(size[s] / size.sum() * 100, 1),
                    "revenue_share_%": summary.loc[s, "revenue_share_%"] if VALUE_COLUMN else np.nan,
                    "stability": seg_stability[s],
                    "profile": descriptions[s], "suggested_action": action})
action_table = pd.DataFrame(actions).set_index("segment")
SEG_NAMES = action_table["suggested_name"].to_dict()
with pd.option_context("display.max_colwidth", 90):
    display(action_table)

## Section 8: A 2-D Map of the Segments (PCA)

PCA squeezes all clustering columns into two summary axes so we can draw the customers (Lesson 6.7).
**How to read it:** clear blobs = well-separated segments; some overlap is normal, because the real separation happens in all columns at once, not just these two. Don't over-interpret the axes themselves.

In [ ]:
# ============================================================
# PCA SCATTER
# ============================================================
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pts = pca.fit_transform(Z)
centres = pca.transform(seg_model.named_steps["kmeans"].cluster_centers_)
plot_idx = np.arange(len(pts)) if len(pts) <= 5_000 else rng.choice(len(pts), 5_000, replace=False)  # keep big plots fast

plt.figure(figsize=(8, 6))
for s in SEGMENTS:
    m = plot_idx[labels[plot_idx] == s]
    plt.scatter(pts[m, 0], pts[m, 1], s=12, alpha=0.55, label=f"{s}: {SEG_NAMES[s]} ({size[s]:,})")
plt.scatter(centres[:, 0], centres[:, 1], marker="X", s=220, c="black", label="segment centres")
ev = pca.explained_variance_ratio_ * 100
plt.xlabel(f"PC1 ({ev[0]:.0f}% of variation)"); plt.ylabel(f"PC2 ({ev[1]:.0f}% of variation)")
plt.title("Customer segments (PCA map)"); plt.legend(fontsize=8, loc="best"); plt.tight_layout(); plt.show()
print(f"The two axes capture {ev.sum():.0f}% of the variation in the clustering columns.")

## Section 9: A Rulebook Staff Can Use (Shallow Decision Tree)

K-means needs code to assign a new customer. A **shallow decision tree** trained to mimic the segments, using the **original** columns, turns them into a few if-then rules anyone can apply in a spreadsheet or CRM (Lesson 6.8).

- **Agreement %**: how often the rulebook gives the same segment as K-means. Above ~85% means the rules are a faithful summary.
- We also check agreement on customers the tree didn't learn from, so the number isn't flattering itself.

In [ ]:
# ============================================================
# RULEBOOK TREE (depth 3, original units)
# ============================================================
RULE_DEPTH = 3                                   # 2 = simplest; 3 = usually enough
X_filled = X.fillna(X.median())                  # trees need no blanks; median like the pipeline
min_leaf = max(10, int(0.02 * len(X)))           # no rule for a handful of customers

# Honest agreement: learn rules on 75%, check on the other 25%
Xa, Xb, la, lb = train_test_split(X_filled, labels, test_size=0.25, random_state=RANDOM_STATE, stratify=labels)
holdout_agree = DecisionTreeClassifier(max_depth=RULE_DEPTH, min_samples_leaf=min_leaf,
                                       random_state=RANDOM_STATE).fit(Xa, la).score(Xb, lb)

tree = DecisionTreeClassifier(max_depth=RULE_DEPTH, min_samples_leaf=min_leaf, random_state=RANDOM_STATE)
tree.fit(X_filled, labels)
RULEBOOK_AGREEMENT = tree.score(X_filled, labels)
print(f"The rulebook agrees with K-means on {RULEBOOK_AGREEMENT:.0%} of customers "
      f"({holdout_agree:.0%} on customers it didn't learn from).")
print("Reading: " + ("faithful summary ✓" if holdout_agree >= 0.85 else
                     "decent summary; a few customers sit between segments" if holdout_agree >= 0.70 else
                     "rough summary only; try RULE_DEPTH = 3 or fewer segments"))
print()
class_names = [f"{c}: {SEG_NAMES[c]}" for c in tree.classes_]
print(export_text(tree, feature_names=CLUSTER_COLUMNS, class_names=class_names, decimals=1))

In [ ]:
# ============================================================
# RULEBOOK AS A TABLE: one plain-English rule per leaf
# ============================================================
# If both branches of a split end in the SAME segment, the split only sharpens
# purity, so we merge them into one simpler rule.
t = tree.tree_
def leaf_classes(node):
    if t.children_left[node] == -1:
        return {t.value[node][0].argmax()}
    return leaf_classes(t.children_left[node]) | leaf_classes(t.children_right[node])

def leaf_rules(node=0, conds=()):
    if t.children_left[node] == -1 or len(leaf_classes(node)) == 1:   # a leaf (or merged branch)
        counts = t.value[node][0]
        seg = tree.classes_[counts.argmax()]
        yield {"rule": " AND ".join(conds) or "everyone", "segment": seg, "segment_name": SEG_NAMES[seg],
               "customers": int(t.n_node_samples[node]), "purity_%": round(counts.max() / counts.sum() * 100)}
        return
    f, thr = CLUSTER_COLUMNS[t.feature[node]], t.threshold[node]
    yield from leaf_rules(t.children_left[node], conds + (f"{f} <= {thr:,.1f}",))
    yield from leaf_rules(t.children_right[node], conds + (f"{f} > {thr:,.1f}",))

rulebook = pd.DataFrame(list(leaf_rules())).sort_values(["segment", "customers"], ascending=[True, False])
with pd.option_context("display.max_colwidth", 120):
    display(rulebook.reset_index(drop=True))
print("Purity = share of customers matching that rule who really are in the named segment.")
print("Copy these rules into a spreadsheet or CRM to tag new customers without running code.")

## Section 10: Small Data vs. Large Data: What Changes

**Small data (hundreds to a few thousand customers):**
- Keep K small (3–5) and `MIN_SEGMENT_PCT` at 5–10% so no segment is a handful of people
- Standard K-means with many restarts (`n_init=20`, automatic in small mode)
- Trust the **stability checks** above before anything else; if resample ARI is under 0.6, simplify
- Fewer, well-chosen columns (3–6) beat many weak ones; consider simple RFM rules as a benchmark (Lesson 6.2)
- Present segments as a first draft, then confirm with the people who know the customers

**Large data (tens of thousands and up):**
- `MODE = "large"` switches to **MiniBatchKMeans** (much faster) and a **sampled** silhouette
- You can fit on a random sample (e.g. 50,000 customers) and assign everyone else with `seg_model.predict()`
- Silhouette differences between K values become tiny; lean even more on business usability
- Watch for a very large "everyone else" segment: it often hides useful sub-groups. Try clustering just that segment again
- Re-score monthly; with lots of data, segment **movement** (VIP → Lapsing) is itself a powerful signal

In [ ]:
# ============================================================
# SAVE (optional): segment labels + the full pipeline
# The pipeline contains prep + K-means, so new customers with the same
# columns can be scored later with:  bundle["model"].predict(new_df[bundle["columns"]])
# ============================================================
import joblib, datetime

out = df[[ID_COLUMN]].copy() if ID_COLUMN and ID_COLUMN in df.columns else pd.DataFrame(index=df.index)
out["segment"] = labels
out["segment_name"] = out["segment"].map(SEG_NAMES)
out["suggested_action"] = out["segment"].map(action_table["suggested_action"])

if SAVE_OUTPUTS:
    out.to_csv("customer_segments.csv", index=False)
    joblib.dump({"model": seg_model, "columns": CLUSTER_COLUMNS, "names": SEG_NAMES,
                 "actions": action_table["suggested_action"].to_dict(), "k": K,
                 "trained_on_rows": len(df), "trained_at": datetime.date.today().isoformat()},
                "segmentation_model.joblib")
    print("Saved: customer_segments.csv and segmentation_model.joblib")

# How to use it next month (example on the first 5 customers):
bundle = {"model": seg_model, "columns": CLUSTER_COLUMNS}
print("Example scoring:", bundle["model"].predict(df[CLUSTER_COLUMNS].head()).tolist())
out.head()

In [ ]:
# ============================================================
# ONE-SCREEN SUMMARY
# ============================================================
print(f"Customers segmented: {len(df):,} | K = {K} | silhouette {best_sil:.2f} | "
      f"stability (bootstrap ARI) {STABILITY_SCORE:.2f} | rulebook agreement {RULEBOOK_AGREEMENT:.0%}")
for s in SEGMENTS:
    r = action_table.loc[s]
    rev_txt = f", {r['revenue_share_%']:.0f}% of revenue" if VALUE_COLUMN else ""
    print(f"  {s}. {r['suggested_name']:<18} {r['share_%']:>5.1f}% of customers{rev_txt} -> {r['suggested_action']}")

## Final Checklist

**Data**
- [ ] Clustering columns match the decision the segments will drive (3–8 columns, each with a reason)
- [ ] No IDs or outcome columns in `CLUSTER_COLUMNS`; outcomes are in `PROFILE_COLUMNS`
- [ ] Skewed money/count columns are in `LOG_COLUMNS`; everything is scaled (done by the pipeline)

**Segmentation**
- [ ] K chosen with all three lenses: elbow, silhouette, and segment size / business usability
- [ ] Stability checked: resample ARI ≥ 0.6 (ideally ≥ 0.8) and no segment far below the others
- [ ] Profiles read in original units and against the average (index)
- [ ] Segments also differ on profile-only outcomes (margin, satisfaction), not just on their inputs

**Business**
- [ ] Every segment has a name, one owner, and one action
- [ ] Rulebook reviewed with the team; agreement % is acceptable
- [ ] A test planned before full rollout (control group per segment)
- [ ] Refresh schedule set (monthly or quarterly)

## What To Do Next

- **Give each segment an owner and one action** from the action table; keep messages and offers different per segment
- **Protect the high-value segment first:** it usually holds a big share of revenue in a small share of customers
- **Win back lapsing customers early:** the longer they're gone, the harder they are to recover
- **Test before rolling out:** for each segment, hold back a random 10–20% as a control group and compare spend after 6–8 weeks
- **Refresh monthly or quarterly:** re-score customers with the saved pipeline and track movement between segments (*New → Core* is good, *VIP → Lapsing* is an alarm)
- **Connect to other modules:** use the segment as an input to the churn model (Module 9 notebook), or build a separate churn model per segment if they behave very differently